<a href="https://colab.research.google.com/github/Moquiuti/LangChainePython/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai pypdf faiss-cpu python-dotenv

In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("python-ai-integrada")
os.environ["GOOGLE_API_KEY"] = api_key

print("Chave carregada com sucesso!" if api_key else "Chave não encontrada.")

In [ ]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
arquivos_pdf = [
    "/content/documentos/GTB_standard_Nov23.pdf",
    "/content/documentos/GTB_gold_Nov23.pdf",
    "/content/documentos/GTB_platinum_Nov23.pdf",
]

documentos = []
for caminho in arquivos_pdf:
    loader = PyPDFLoader(caminho)
    documentos.extend(loader.load())

print(f"Total de documentos carregados: {len(documentos)}")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documentos)

print(f"Total de chunks gerados: {len(chunks)}")
print(chunks[0].page_content[:500])

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store criado com sucesso.")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
consulta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

docs_relevantes = retriever.invoke(consulta)

for i, doc in enumerate(docs_relevantes, start=1):
    print(f"\n--- Documento {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Você é um assistente que responde perguntas com base apenas no contexto abaixo.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
""")

In [ ]:
def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
cadeia_rag = (
    {
        "contexto": retriever | formatar_contexto,
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

resposta = cadeia_rag.invoke(pergunta)
print(resposta)

In [ ]:
pergunta_2 = "Quais benefícios eu tenho com o cartão platinum em viagens?"
resposta_2 = cadeia_rag.invoke(pergunta_2)
print(resposta_2)

formato memrag.py

In [ ]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def main():
    arquivos_pdf = [
        "documentos/GTB_standard_Nov23.pdf",
        "documentos/GTB_gold_Nov23.pdf",
        "documentos/GTB_platinum_Nov23.pdf",
    ]

    documentos = []
    for caminho in arquivos_pdf:
        loader = PyPDFLoader(caminho)
        documentos.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documentos)

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.3
    )

    prompt = ChatPromptTemplate.from_template("""
    Você é um assistente que responde perguntas com base apenas no contexto abaixo.

    Contexto:
    {contexto}

    Pergunta:
    {pergunta}

    Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
    """)

    cadeia_rag = (
        {
            "contexto": retriever | formatar_contexto,
            "pergunta": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"
    resposta = cadeia_rag.invoke(pergunta)

    print("\nPergunta:")
    print(pergunta)
    print("\nResposta:")
    print(resposta)

if __name__ == "__main__":
    main()

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai pypdf faiss-cpu sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
consulta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

docs_relevantes = retriever.invoke(consulta)

for i, doc in enumerate(docs_relevantes, start=1):
    print(f"\n--- Documento {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Você é um assistente que responde perguntas com base apenas no contexto abaixo.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
""")

In [ ]:
def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
cadeia_rag = (
    {
        "contexto": retriever | formatar_contexto,
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

resposta = cadeia_rag.invoke(pergunta)
print(resposta)

In [ ]:
pergunta_2 = "Quais benefícios eu tenho com o cartão platinum em viagens?"
resposta_2 = cadeia_rag.invoke(pergunta_2)
print(resposta_2)